# Defter 23B — Transfer to a warmer period and sensitivity to the model structure
**Theory-guided AI reveals contrasting effects of a century of climate change on tree growth** · Prof. Dr. İlker Ercanlı

**Ne yapar (TR):** Hakem değerlendirmesinin 4. ve 7. maddelerine cevap verir; iki yeni eğitim içerir:

| Bölüm | Soru | Eğitim | Çıktı (`SONUC_23B/`) |
|---|---|---|---|
| B1 | Yalnızca 1902–1960 halkalarıyla eğitilen ağ, sıcak dönemi (1990–2024) ne kadar iyi tahmin ediyor ve aynı atfı veriyor mu? | 5 tohum | `B1_erken_beceri.csv`, `B1_erken_atif.csv` |
| B2 | Minimum yasası yerine **çarpım** kuralı (C = Σ ω g_E g_T g_M g_D) kullanılırsa beceri ve atıf değişiyor mu? | 4 kat × 5 tohum | `B2_carpim_beceri.csv`, `B2_carpim_atif.csv` |
| B3 | Üç model ailesinin (ana, erken, çarpım) 1990–2019 atfı aynı sahalarda, halka tahminiyle birlikte | — | `B3_karsilastirma.csv`, `B3_onyil.csv`, `B3_alan.csv` |

Mimari, eğitim ayarları ve iklim girdileri Defter 20 ile aynı (2 × 128). Atıf Defter 21 ile aynı yöntemle yapılır.
**Önce:** Defter 20, 21 ve 23A bitmiş olmalı.
**Çalışma zamanı:** **L4** GPU, yüksek RAM → "Tümünü çalıştır". Kesilirse yeniden "Tümünü çalıştır": biten modeller ve
biten atıf dosyaları atlanır. Süre: B1 ~15 dk, B2 ~60–90 dk (20 model), atıf ~30 dk.
**Bitiş satırı:** `==== DEFTER 23B TAMAM ====`

In [ ]:
# --- Setup ---
import os, json, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:                                   # local test outside Colab
    ANA = os.environ.get('ITRDB_AI', '.')
KAYNAK = os.path.join(ANA, 'SONUC_20')                # final 20-member model of Defter 20
K21 = os.path.join(ANA, 'SONUC_21')                   # its saved attribution (Defter 21)
CIKTI = os.path.join(ANA, 'SONUC_23B'); os.makedirs(CIKTI, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(no GPU: Runtime -> change runtime type -> GPU)')

AYAR = dict(
    batch=32768, max_epoch=60, sabir=5, lr=2e-3, lr_agac=1e-2, wd=1e-4,
    J=8, emb=8,
    arama_katman=[1, 2, 4, 6, 8],                    # architecture search of the site-parameter network (stage A)
    arama_gizli=[32, 64, 128, 256, 512],
    arama_tolerans=0.002,                            # pick the smallest network within this early-stopping NLL of the best
    mimari=None,                                     # e.g. [128, 2] to skip the search and fix the architecture
    modeller=['teori_vpd_opt_re', 'teori_vpd_opt_spei_re', 'iklimsiz_re'],
    ana_model='teori_vpd_opt_spei_re',               # model used for the search
    n_tohum=5,                                       # ensemble members (epistemic uncertainty)
    saha_tohum=1,                                    # seeds for the unseen-site test
    asamalar=['arama', 'ana', 'saha', 'topluluk'],   # stages to run; drop any to skip
    takson_dusur=0.10, desc_gurultu=0.2, r0_bas_cm=2.0, sifir_mm=0.005,
    blok_yil=5, n_kat=4, bolme_tohum=2026,           # year-block split: 5-year blocks, 4 folds of ~25% of years
    ic_blok_oran=0.10,                               # share of training blocks held out for early stopping (unseen years)
    agac_onsel=0.3,                                  # prior SD of the tree effect (log scale)
    hizli_deneme=False,                              # True: small subset and 2 epochs, to check the notebook end to end
    model='teori_vpd_opt_re',                        # final model (Defter 20)
    yil_son=2019,                                    # ISIMIP3a obsclim/counterclim end year
    erken_son=1960,                                  # B1: train on rings up to this year
    test_bas=1990,                                   # B1: test on rings from this year
    n_tohum_erken=5, n_tohum_carpim=5,               # ensemble sizes of B1 and B2
    analiz_son=None,                                 # folder of the draft-10 ring tables; None = search
)
AYAR.update(json.loads(os.environ.get('D23B_AYAR', '{}')))   # override only for local testing
if AYAR['hizli_deneme']: AYAR['max_epoch'] = min(AYAR['max_epoch'], 2)
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

In [ ]:
# --- Data (identical preparation to Defter 2) ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1

clim = d['clim']; soil = d['soil']; gE = d['gE']; NY = clim.shape[1]
YIL1 = YIL0 + NY - 1
# rings need the previous year's climate, so 1901 rings are dropped (as in Defter 2)
k = (d['ring_year'] > YIL0) & (d['ring_year'] <= YIL1)
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k].astype(np.int64), torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
R_site = ser_site[R_ser]; R_tree = ser_tree[R_ser]
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# monthly climate per site: T, log(1+P), soil moisture M, VPD, growing-season weight gE (+ CWD, SPEI-like, below)
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
# --- additional water-stress variables (computed from the same CRU TS and bucket-model data; no download) ---
# climatic water deficit: CWD = PET - AET, with AET = PET x M / Mmax (supply-limited evapotranspiration of the
# VS-Lite bucket, Mmax = 0.76); in units of 100 mm per month
MMAX = 0.76
pet = np.maximum(np.nan_to_num(clim[..., 2]), 0)
cwd = (pet * (1 - np.clip(soil, 0, MMAX) / MMAX) / 100).reshape(S, NY * 12)
# SPEI-like index: 12-month sum of P - PET, standardized per site and calendar month against 1901-1960
wb = (np.nan_to_num(clim[..., 1]) - pet).reshape(S, NY * 12)
cs = np.cumsum(wb, axis=1); wb12 = np.full_like(cs, np.nan); wb12[:, 11:] = cs[:, 11:] - np.concatenate([np.zeros((S, 1)), cs[:, :-12]], 1)
wb12 = wb12.reshape(S, NY, 12); ref = wb12[:, :60]
z12 = (wb12 - np.nanmean(ref, 1, keepdims=True)) / (np.nanstd(ref, 1, keepdims=True) + 1e-6)
z12 = np.nan_to_num(np.clip(z12, -4, 4)).reshape(S, NY * 12)
ozel = np.concatenate([ozel, cwd[..., None].astype(np.float32), z12[..., None].astype(np.float32)], -1)
print(f'Water deficit: median {np.median(cwd)*100:.0f} mm/month; SPEI-like index range {z12.min():.1f} to {z12.max():.1f}')
del pet, cwd, wb, cs, wb12, ref, z12
n_nan = int(np.isnan(ozel).sum())
if n_nan: print(f'WARNING: {n_nan} missing climate values set to the site-month mean')
if n_nan:
    nk = ozel.shape[-1]
    ort = np.nanmean(ozel.reshape(S, NY, 12, nk), axis=1, keepdims=True)
    o4 = ozel.reshape(S, NY, 12, nk); o4[:] = np.where(np.isnan(o4), ort, o4); ozel = o4.reshape(S, NY * 12, nk)
OZ = t(ozel); del ozel

desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
env = d['env']; ENV = t((env - env.mean(0)) / (env.std(0) + 1e-6), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
IC = t(d['site_ic_dogrulama'], torch.bool)            # 10% of sites reserved for early stopping (fixed in Defter 1)
print(f'{NR:,} rings, {NSER:,} series, {NTREE:,} trees, {S} sites, years {YIL0+1}-{YIL1}')

def pencere(idx):
    """24-month climate window (previous January - current December) for rings idx: [B, 24, 5]"""
    s = R_site[idx]; bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

def girdiler(idx):
    X, s = pencere(idx)
    return dict(X=X, rp=R_rp[idx], tree=R_tree[idx], g=site_genus[s], sp=site_species[s], desc=DESC[s],
                env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s])

In [ ]:
# --- Year-block split: 5-year blocks; within each run of 4 consecutive blocks the folds are permuted,
#     so every fold covers the whole period (early, mid-century and the recent warm decades) ---
yillar = np.arange(YIL0 + 1, YIL1 + 1)
blok = (yillar - yillar[0]) // AYAR['blok_yil']; nb = int(blok.max()) + 1
rng = np.random.default_rng(AYAR['bolme_tohum']); kat_blok = np.empty(nb, int)
for b0 in range(0, nb, AYAR['n_kat']):
    ix = np.arange(b0, min(b0 + AYAR['n_kat'], nb)); kat_blok[ix] = rng.permutation(AYAR['n_kat'])[:len(ix)]
yk = np.full(YIL1 + 2, -1, np.int64); yk[yillar] = kat_blok[blok]
YK = t(yk, torch.long)
tablo = pd.DataFrame(dict(year=yillar, block=blok, fold=kat_blok[blok]))
SITE_KAT = t(d['site_blok_kat'], torch.long)          # 10-degree spatial blocks in 5 groups (fixed in Defter 1)
R_kat = YK[R_yil]
for kk in range(AYAR['n_kat']):
    yy = tablo[tablo.fold == kk].year.to_numpy()
    print(f'fold {kk}: {len(yy)} years ({len(yy)/len(yillar):.0%}), {(R_kat == kk).float().mean().item():.1%} of rings; '
          f'blocks starting ' + ', '.join(str(y) for y in yy[::AYAR["blok_yil"]]))

bk = np.full(YIL1 + 2, -1, np.int64); bk[yillar] = blok
R_blok = t(bk, torch.long)[R_yil]

def bolunme(kat):
    """Test: this fold's years (~25%) at all sites. Early stopping: ~10% of the remaining 5-year blocks, drawn at
    random, at all sites (unseen years, as in the test). Train: all other years at all sites.
    No test year is used for training or for early stopping."""
    egit_blok = np.unique(blok[kat_blok[blok] != kat])
    n_ic = max(2, int(round(AYAR['ic_blok_oran'] * len(egit_blok))))
    ic_blok = np.random.default_rng(AYAR['bolme_tohum'] + 100 + kat).choice(egit_blok, n_ic, replace=False)
    test = R_kat == kat; ic = torch.isin(R_blok, t(ic_blok, torch.long))
    egit = (~test & ~ic).nonzero().squeeze(1); dog = (~test & ic).nonzero().squeeze(1); tst = test.nonzero().squeeze(1)
    return _alt(egit, dog, tst)

def bolunme_erken(kat=0):
    """B1: train on years up to AYAR['erken_son'] (early stopping on ~10% of those 5-year blocks, at all sites);
    test on years from AYAR['test_bas'] at all sites. Rings of the years in between are not used."""
    erken_blok = np.unique(blok[yillar <= AYAR['erken_son']])
    n_ic = max(2, int(round(AYAR['ic_blok_oran'] * len(erken_blok))))
    ic_blok = np.random.default_rng(AYAR['bolme_tohum'] + 500 + kat).choice(erken_blok, n_ic, replace=False)
    e = R_yil <= AYAR['erken_son']; ic = torch.isin(R_blok, t(ic_blok, torch.long))
    return _alt((e & ~ic).nonzero().squeeze(1), (e & ic).nonzero().squeeze(1), (R_yil >= AYAR['test_bas']).nonzero().squeeze(1))

def bolunme_saha(kat):
    """Unseen-site test: test = all years at the sites of spatial group `kat` (10-degree blocks); early stopping =
    the fixed 10% validation sites outside the test group (also unseen sites); train = all other sites, all years.
    Trees at test sites were never seen, so their tree effect stays at its prior mean (zero)."""
    ts = SITE_KAT[R_site] == kat; ic = IC[R_site] & ~ts
    return _alt((~ts & ~ic).nonzero().squeeze(1), ic.nonzero().squeeze(1), ts.nonzero().squeeze(1))

def _alt(egit, dog, tst):
    if AYAR['hizli_deneme']:
        g = torch.Generator(device='cpu').manual_seed(0)
        sec = lambda x, n: x[torch.randperm(len(x), generator=g)[:n].to(DEV)]
        egit, dog = sec(egit, 200_000), sec(dog, 20_000)
    return egit, dog, tst

In [ ]:
# --- Model: theory-embedded network (equations 1-3 of the manuscript) with options ---
class MLP(nn.Module):
    """Site-parameter network with `katman` hidden layers of `gizli` units (residual blocks beyond the first)."""
    def __init__(s, n_in, gizli, katman, n_out):
        super().__init__()
        s.gir = nn.Linear(n_in, gizli)
        s.bloklar = nn.ModuleList([nn.Sequential(nn.LayerNorm(gizli), nn.Linear(gizli, gizli), nn.GELU()) for _ in range(katman - 1)])
        s.cik = nn.Linear(gizli, n_out)
        with torch.no_grad(): s.cik.weight.mul_(0.1); s.cik.bias.zero_()
    def forward(s, x):
        h = F.gelu(s.gir(x))
        for b in s.bloklar: h = h + b(h)
        return s.cik(h)

class SiteAgi(nn.Module):
    """Genus/species embeddings + site descriptors + site environment (soil, topography, biome, ecoregion, Koppen)."""
    def __init__(s, n_out, gizli, katman):
        super().__init__()
        e = AYAR['emb']
        s.ge = nn.Embedding(NG + 1, e); s.se = nn.Embedding(NS + 1, e)              # last index = unknown taxon
        s.b = nn.Embedding(NB + 1, 4); s.e = nn.Embedding(NE + 1, 4); s.k = nn.Embedding(NK + 1, 4)
        s.mlp = MLP(2 * e + DESC.shape[1] + ENV.shape[1] + 12, gizli, katman, n_out)
    def forward(s, G, gizle=None):
        g, sp, b, e, k = G['g'], G['sp'], G['b'], G['e'], G['k']
        if gizle is not None:                                                   # taxon dropout: learn the 'unknown' embedding
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        if s.training:                                                          # 10% masking of the categorical environment
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        dn = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if s.training else 0)
        en = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if s.training else 0)
        return s.mlp(torch.cat([s.ge(g), s.se(sp), dn, en, s.b(b), s.e(e), s.k(k)], 1))

def monoton(x, a, c, s):
    """Equation (3): non-negative mixture of logistic functions, rising from 0 to 1. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

class TeoriModeli19(nn.Module):
    def __init__(s, vpd=True, opt=False, iklim=True, agac=False, cwd=False, spei=False, gizli=128, katman=2, carpim=False):
        super().__init__()
        J = AYAR['J']; s.J, s.vpd, s.opt, s.iklim, s.agac, s.cwd, s.spei, s.carpim = J, vpd, opt, iklim, agac, cwd, spei, carpim
        n = 3 + 6 * J + 24 + (3 * J if vpd else 0) + (2 if opt else 0) + (3 * J if cwd else 0) + (3 * J if spei else 0) + 3 + 1
        s.ag = SiteAgi(n, gizli, katman)
        s.Tbaz = torch.linspace(-5, 25, J, device=DEV); s.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)
        s.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        s.Wbaz = torch.linspace(0.05, 1.5, J, device=DEV); s.Zbaz = torch.linspace(-2.5, 1.5, J, device=DEV)
        s.u = nn.Parameter(torch.zeros(NTREE, device=DEV))                     # tree effect (used only when agac=True)
        s.m_logit = math.log((2 / 3) / (1 - 2 / 3))

    def parametreler(s, G, gizle=None):
        o = s.ag(G, gizle); J = s.J; p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm^2
        p['m'] = torch.sigmoid(o[:, 2] + s.m_logit)                            # starts at 2/3 (metabolic scaling)
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], s.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], s.Mbaz + 0.1 * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if s.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], s.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.cwd:   # climatic water deficit (100 mm/month); response decreases with deficit
            p['aW'], p['cW'], p['sW'] = o[:, i:i+J], s.Wbaz + 0.2 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.spei:  # 12-month standardized water balance; response increases with wetter conditions
            p['aZ'], p['cZ'], p['sZ'] = o[:, i:i+J], s.Zbaz + 0.5 * torch.tanh(o[:, i+J:i+2*J]), 0.2 + 0.5 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.opt:   # temperature optimum: growth declines above a site-specific upper threshold (starts near 25 °C)
            p['cH'], p['sH'] = 25 + 7 * torch.tanh(o[:, i]), 0.5 + 2 * F.softplus(o[:, i+1]); i += 2
        p['sig0'], p['sigC'], p['sigR'] = o[:, i] - 0.3, 0.1 * o[:, i+1], 0.1 * o[:, i+2]; i += 3   # heteroscedastic SD
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1
        return p

    def forward(s, idx, gizle=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = s.parametreler(G, gizle)
        r = p['r0'] + G['rp']; B = math.pi * r ** 2
        x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
        Gp = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20       # equation (1), smoothed
        T, M, D, gEk = X[..., 0], X[..., 2], X[..., 3], X[..., 4]
        if s.iklim:
            gT = monoton(T, p['aT'], p['cT'], p['sT'])
            if s.opt: gT = gT * (1 - torch.sigmoid((T - p['cH'][:, None]) / p['sH'][:, None]))
            gM = monoton(M, p['aM'], p['cM'], p['sM'])
            sinir = torch.minimum(gT, gM)
            gD = 1 - monoton(D, p['aD'], p['cD'], p['sD']) if s.vpd else torch.ones_like(gT)
            sinir = torch.minimum(sinir, gD)                                   # Liebig: min(gT, gM, gD), equation (2)
            if s.carpim: sinir = gT * gM * gD                                  # alternative structure: multiplicative limitation
            if s.cwd: sinir = torch.minimum(sinir, 1 - monoton(X[..., 5], p['aW'], p['cW'], p['sW']))
            if s.spei: sinir = torch.minimum(sinir, monoton(X[..., 6], p['aZ'], p['cZ'], p['sZ']))
        else:
            gT = gM = gD = sinir = torch.ones_like(T)                          # climate-free ablation
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        u = s.u[G['tree']] if s.agac else torch.zeros_like(C)
        BAI = Gp * C * torch.exp(u)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                     # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        ls = p['sig0'] + p['sigC'] * torch.log(C) + p['sigR'] * torch.log(r)  # aleatoric SD varies with climate and size
        sicak = (T > p['cH'][:, None]) if s.opt else torch.zeros_like(T, dtype=torch.bool)
        return mu, ls, dict(gT=gT, gM=gM, gD=gD, C=C, G=Gp, w=p['omega'] * gEk, sicak=sicak)

def yap(ad, gizli, katman):
    return TeoriModeli19(vpd=True, opt='_opt' in ad, iklim=not ad.startswith('iklimsiz'), agac=ad.endswith('_re'),
                         cwd='_cwd' in ad, spei='_spei' in ad, gizli=gizli, katman=katman, carpim='_carpim' in ad).to(DEV)

def ag_param(m):
    return sum(p.numel() for n, p in m.named_parameters() if n != 'u')

In [ ]:
# --- Training and evaluation ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm']); LOG_ESIK = math.log(0.01)
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

def nll(mu, ls, y):
    """Gaussian NLL of log ring width; zero rings are left-censored below the 0.01 mm resolution."""
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, bs=131072):
    model.eval(); MU, LS = [], []
    for i in range(0, len(idx), bs):
        mu, ls, _ = model(idx[i:i+bs]); MU.append(mu); LS.append(ls)
    model.train(); return torch.cat(MU), torch.cat(LS)

def yillik_sinyal(idx, mu, y):
    """Median site correlation between observed and predicted growth anomalies, averaged per site and year
    (sites with >= 8 test years). 'tum': each series centred on its test-period mean, so slow size/age trends and
    differences between blocks count. 'blok_ici': each series centred within each 5-year test block, which removes
    trends and levels and keeps only year-to-year (high-frequency, mainly climatic) variation."""
    nz = (y > LOG_Y_SIFIR + 1e-6).cpu().numpy()
    df = pd.DataFrame(dict(ser=R_ser[idx].cpu().numpy(), s=R_site[idx].cpu().numpy(), yil=R_yil[idx].cpu().numpy(),
                           b=R_blok[idx].cpu().numpy(), o=y.cpu().numpy(), p=mu.float().cpu().numpy()))[nz]
    sonuc = {}
    for ad, grup in [('tum', ['ser']), ('blok_ici', ['ser', 'b'])]:
        a = df[['s', 'yil']].copy()
        a['o'] = df.o - df.groupby(grup).o.transform('mean'); a['p'] = df.p - df.groupby(grup).p.transform('mean')
        sy = a.groupby(['s', 'yil'])[['o', 'p']].mean().reset_index()
        r = sy.groupby('s').apply(lambda g: g.o.corr(g.p) if len(g) >= 8 else np.nan, include_groups=False).dropna()
        sonuc[ad] = (float(r.median()), int(len(r)))
    return sonuc


def is_adi(bolme, kat, ad, gizli, katman, tohum):
    return f'{bolme}{kat}__{ad}__h{gizli}x{katman}__s{tohum}'

def egit(bolme, kat, ad, gizli, katman, tohum, kaydet=True):
    """Trains one job and evaluates it on its test set. bolme: 'arama' or 'yil' (year folds), 'saha' (site groups)."""
    is_ad = is_adi(bolme, kat, ad, gizli, katman, tohum)
    kl = os.path.join(CIKTI, is_ad); os.makedirs(kl, exist_ok=True)
    if os.path.exists(os.path.join(kl, 'bitti.json')):
        return json.load(open(os.path.join(kl, 'bitti.json')))
    torch.manual_seed(tohum); np.random.seed(tohum)
    egt, dog, tst = bolunme_saha(kat) if bolme == 'saha' else (bolunme_erken(kat) if bolme == 'erken' else bolunme(kat))
    model = yap(ad, gizli, katman)
    ag = [p for n, p in model.named_parameters() if n != 'u']
    opt = torch.optim.AdamW([{'params': ag, 'lr': AYAR['lr'], 'weight_decay': AYAR['wd']},
                             {'params': [model.u], 'lr': AYAR['lr_agac'] if model.agac else 0.0, 'weight_decay': 0.0}])
    kayit = os.path.join(kl, 'son.pt'); bas_ep, en_iyi, sabir, gecmis = 0, float('inf'), 0, []
    if os.path.exists(kayit):
        ck = torch.load(kayit, map_location=DEV)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        bas_ep, en_iyi, sabir, gecmis = ck['epoch'] + 1, ck['en_iyi'], ck['sabir'], ck['gecmis']
        print(f'{is_ad}: resuming from epoch {bas_ep}')
    n_egt = len(egt); t_bas = time.time()
    for ep in range(bas_ep, AYAR['max_epoch']):
        t0 = time.time(); kar = egt[torch.randperm(n_egt, device=DEV)]; top = 0.
        for i in range(0, n_egt, AYAR['batch']):
            b = kar[i:i+AYAR['batch']]
            gz = torch.rand(len(b), device=DEV) < AYAR['takson_dusur']
            mu, ls, _ = model(b, gz)
            kv = nll(mu, ls, hedef(b))
            kayip = kv + ((model.u[R_tree[b]] ** 2).mean() / (2 * AYAR['agac_onsel'] ** 2) * len(b) / n_egt if model.agac else 0.)
            opt.zero_grad(set_to_none=True); kayip.backward()
            torch.nn.utils.clip_grad_norm_(ag, 5.0); opt.step(); top += kv.item() * len(b)
        mu, ls = tahmin(model, dog); v = nll(mu, ls, hedef(dog)).item()
        gecmis.append(dict(epoch=ep, egitim=top / n_egt, dogrulama=v, sn=time.time() - t0))
        if v < en_iyi - 1e-4:
            en_iyi, sabir = v, 0; torch.save(model.state_dict(), os.path.join(kl, 'en_iyi.pt'))
        else:
            sabir += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep, en_iyi=en_iyi, sabir=sabir, gecmis=gecmis), kayit)
        if sabir >= AYAR['sabir']: break
    model.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV))
    mu, ls = tahmin(model, tst); y = hedef(tst); sg = torch.exp(ls)
    nz = y > LOG_Y_SIFIR + 1e-6; e = (mu - y)[nz]
    ys = yillik_sinyal(tst, mu, y)
    if kaydet:
        np.savez_compressed(os.path.join(kl, 'test_tahmin.npz'), mu=mu.cpu().numpy().astype(np.float16),
                            sigma=sg.cpu().numpy().astype(np.float16))
    ozet = dict(is_ad=is_ad, bolme=bolme, kat=kat, model=ad, gizli=gizli, katman=katman, tohum=tohum,
                ag_param=ag_param(model), n_egitim=n_egt, n_dogrulama=len(dog), n_test=len(tst), epoch=len(gecmis),
                sn_epoch=float(np.mean([g['sn'] for g in gecmis])), rmse_log=float(e.pow(2).mean().sqrt()),
                bias_log=float(e.mean()), r2_log=float(1 - e.pow(2).mean() / y[nz].var()), nll=float(nll(mu, ls, y)),
                kapsama90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()),
                yillik_r_medyan=ys['tum'][0], blok_ici_r_medyan=ys['blok_ici'][0], blok_ici_r_n_saha=ys['blok_ici'][1],
                en_iyi_dogrulama=en_iyi, ayar=AYAR, gecmis=gecmis)
    json.dump(ozet, open(os.path.join(kl, 'bitti.json'), 'w'), indent=1)
    os.remove(kayit)
    print(f'DONE {is_ad}: {len(gecmis)} ep, {(time.time()-t_bas)/60:.1f} min | early-stop NLL {en_iyi:.4f} | test R2 {ozet["r2_log"]:.3f}, '
          f'within-block r {ozet["blok_ici_r_medyan"]:.3f}, 90% cov {ozet["kapsama90"]:.3f}')
    return ozet

def tablo_yap(ozetler):
    return pd.DataFrame([{k: v for k, v in o.items() if k not in ('ayar', 'gecmis')} for o in ozetler])

# test indices of each year fold, in a fixed order (needed to assemble out-of-fold predictions)

In [ ]:
# --- Load the final model: 4 year folds x 5 seeds (Defter 20) ---
import glob, gc, re
f_ = os.path.join(KAYNAK, 'P2_secilen_mimari.json')
GIZLI, KATMAN = (json.load(open(f_))['gizli'], json.load(open(f_))['katman']) if os.path.exists(f_) else (128, 2)

def yukle(kl):
    m = yap(AYAR['model'], GIZLI, KATMAN); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m

UYE = []
for kl in sorted(glob.glob(os.path.join(KAYNAK, f'yil*__{AYAR["model"]}__h{GIZLI}x{KATMAN}__s*'))):
    if os.path.exists(os.path.join(kl, 'bitti.json')) and os.path.exists(os.path.join(kl, 'en_iyi.pt')):
        k_, s_ = re.search(r'yil(\d+)__.*__s(\d+)$', kl).groups(); UYE.append(dict(kat=int(k_), tohum=int(s_), kl=kl))
assert UYE, f'No trained models in {KAYNAK}: run Defter 20 first'
MODELLER = [yukle(u['kl']) for u in UYE]
ALT = [i for i, u in enumerate(UYE) if u['tohum'] == min(v['tohum'] for v in UYE)]   # one member per fold (limiting factors, curves)
print(f'Final model {AYAR["model"]} ({KATMAN} x {GIZLI}): {len(MODELLER)} members; subset for Q4-Q5: {len(ALT)}')

@torch.no_grad()
def ongor(m, idx, bs=131072, ek=False):
    """Predictions with the current global climate tensor OZ. ek=True also returns per-ring limiting-factor shares."""
    m.eval(); MU, EK = [], []
    for i in range(0, len(idx), bs):
        mu, ls, e = m(idx[i:i+bs]); MU.append(mu)
        if ek:
            w = e['w']; ws = w.sum(1).clamp_min(1e-9)
            st = torch.stack([e['gT'], e['gM'], e['gD']], -1).argmin(-1)                # 0 temperature, 1 soil moisture, 2 VPD
            pay = lambda msk: (w * msk).sum(1) / ws
            EK.append(torch.stack([pay((st == 0) & ~e['sicak']), pay((st == 0) & e['sicak']), pay(st == 1), pay(st == 2), e['C']], 1))
    return torch.cat(MU), (torch.cat(EK) if ek else None)

# site-level groups used throughout
site_bolge = np.array(soz['site_bolge'])
gen_ad = np.array(soz['genus'] + ['Unknown'])[np.minimum(d['site_genus'], len(soz['genus']))]
soguk = desc[:, 2] <= np.quantile(desc[:, 2], 0.25); kurak = desc[:, 5] <= np.quantile(desc[:, 5], 0.25)
nemli = desc[:, 5] >= np.quantile(desc[:, 5], 0.75)
GRUP = {'All sites': np.ones(S, bool), 'Coldest 25%': soguk, 'Driest 25%': kurak, 'Wettest 25%': nemli,
        **{f'Region: {b}': site_bolge == b for b in np.unique(site_bolge)}}
cs_ = pd.Series(gen_ad).value_counts()
GRUP_CINS = {f'Genus: {g}': gen_ad == g for g in cs_[cs_ >= 30].index if g != 'Unknown'}
yuzde = lambda x: 100 * (np.exp(x) - 1)
SITE_n = R_site.cpu().numpy(); YIL_n = R_yil.cpu().numpy()
Y_all = hedef(torch.arange(NR, device=DEV)).cpu().numpy(); NZ_all = Y_all > LOG_Y_SIFIR + 1e-6

In [ ]:
# --- Q0. How much did climate change? (a) CRU: 1990-2019 vs 1901-1960; (b) ISIMIP3a: observed minus counterfactual climate ---
# Counterfactual climate as in Defter 11: the ISIMIP3a obsclim - counterclim difference (Mengel et al. 2021 detrending) is applied
# to the CRU climate the model was trained on; soil moisture is recomputed with the same VS-Lite bucket.
def gun_uzunlugu(lat):
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))
def isi_indeksi(T):
    Tclim = np.nanmean(T.reshape(T.shape[0], -1, 12), axis=1)
    return np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
def kova(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for k in range(N):
        Tt, Pt, Lt = T[:, k], P[:, k], L[:, k % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, k] = m
    return M

KL_IS = os.path.join(ANA, 'IKLIM_ISIMIP'); YIL_SON = AYAR['yil_son']
kod2s = {c: i for i, c in enumerate(soz['site_code'])}
def isimip_oku(sen):
    f = os.path.join(KL_IS, f'isimip_{sen}_aylik.csv.gz')
    assert os.path.exists(f), f'Missing {f} (prepared for Defter 11 with ISIMIP_cikar.R)'
    df = pd.read_csv(f, usecols=['studyCode', 'year', 'month', 'tmp', 'pre', 'vpd']); df = df[df.studyCode.isin(list(kod2s))]
    s_ = df.studyCode.map(kod2s).to_numpy(); y_ = df.year.to_numpy() - YIL0; m_ = df.month.to_numpy() - 1
    out = {}
    for v in ['tmp', 'pre', 'vpd']:
        a = np.full((S, NY, 12), np.nan, np.float32); a[s_, y_, m_] = df[v].to_numpy(np.float32); out[v] = a
    return out
OBS = isimip_oku('obsclim'); CFI = isimip_oku('counterclim')
isimip_var = ~np.isnan(OBS['tmp']).all(axis=(1, 2))
dT = np.nan_to_num(OBS['tmp'] - CFI['tmp'])
oranP = np.clip(np.nan_to_num(np.where(OBS['pre'] > 1.0, CFI['pre'] / np.maximum(OBS['pre'], 1e-6), 1.0), nan=1.0), 0.3, 3.0)
dV = np.nan_to_num(OBS['vpd'] - CFI['vpd']); del OBS, CFI; gc.collect()
LATs = d['site_lat'].astype(float)
T_f = np.nan_to_num(clim[..., 0].astype(np.float64)); P_f = np.nan_to_num(clim[..., 1].astype(np.float64)); V_f = np.nan_to_num(clim[..., 3].astype(np.float64))
T_c = T_f - dT; P_c = P_f * oranP; V_c = np.maximum(V_f - dV, 0)
I_tam = isi_indeksi(T_f.reshape(S, -1))
M_f = kova(T_f.reshape(S, -1), P_f.reshape(S, -1), LATs, I_tam); M_c = kova(T_c.reshape(S, -1), P_c.reshape(S, -1), LATs, I_tam)
print('Check: recomputed factual soil moisture vs Defter 1 (max abs diff):', round(float(np.nanmax(np.abs(M_f - soil.reshape(S, -1)))), 4))
OZ_F = OZ
def degistir(kanal):
    Z = OZ_F.clone()
    for c, a in kanal.items():
        a = t(np.asarray(a, np.float32).reshape(S, -1)); Z[..., c] = torch.where(torch.isnan(a), Z[..., c], a)
    return Z
# counterfactual soil moisture as a delta on the factual (Defter 1) soil moisture, so that years without a climate-change
# signal are identical in both scenarios even if the recomputed bucket differs slightly from Defter 1
M_cd = np.clip(soil.reshape(S, -1) + (M_c - M_f), 0.01, 0.76)
OZ_C = degistir({0: T_c, 1: np.log1p(np.maximum(P_c, 0)), 2: M_cd, 3: V_c})      # all channels counterfactual
OZ_T = degistir({0: T_c}); OZ_W = degistir({2: M_cd}); OZ_D = degistir({3: V_c})  # one channel at a time
SENARYO = {'F': OZ_F, 'C': OZ_C, 'T': OZ_T, 'W': OZ_W, 'D': OZ_D}

In [ ]:
# --- B1. Train on 1902-1960 only, test on 1990-2024 (transfer to a warmer period); B2. product instead of minimum ---
GIZLI, KATMAN = GIZLI, KATMAN                      # architecture of the final model (read from P2_secilen_mimari.json above)
AD_ERKEN, AD_CARPIM = AYAR['model'], AYAR['model'].replace('_re', '_carpim_re')
B1 = [egit('erken', 0, AD_ERKEN, GIZLI, KATMAN, tohum, kaydet=False) for tohum in range(AYAR['n_tohum_erken'])]
B2 = [egit('yil', kat, AD_CARPIM, GIZLI, KATMAN, tohum, kaydet=False) for tohum in range(AYAR['n_tohum_carpim']) for kat in range(AYAR['n_kat'])]
b1 = tablo_yap(B1); b2 = tablo_yap(B2)

# skill of the main model on the same late years, from its out-of-fold predictions (Defter 20)
oof = torch.as_tensor(np.load(os.path.join(KAYNAK, f'OOF_{AYAR["model"]}.npz'))['mu'].astype(np.float32), device=DEV)
def beceri(idx, mu):
    y = hedef(idx); nz = y > LOG_Y_SIFIR + 1e-6; e = (mu - y)[nz]; ys = yillik_sinyal(idx, mu, y)
    return dict(n_rings=int(len(idx)), r2_log=float(1 - e.pow(2).mean() / y[nz].var()), bias_log=float(e.mean()),
                rmse_log=float(e.pow(2).mean().sqrt()), within_block_r=ys['blok_ici'][0], n_sites=ys['blok_ici'][1])
gec = (R_yil >= AYAR['test_bas']).nonzero().squeeze(1); ara = ((R_yil > AYAR['erken_son']) & (R_yil < AYAR['test_bas'])).nonzero().squeeze(1)
satir = [dict(model='main (year-block cross-validation, out-of-fold)', period=f'{AYAR["test_bas"]}-{YIL1}', **beceri(gec, oof[gec]))]
UYE_E = []
for tohum in range(AYAR['n_tohum_erken']):
    kl = os.path.join(CIKTI, is_adi('erken', 0, AD_ERKEN, GIZLI, KATMAN, tohum))
    m = yap(AD_ERKEN, GIZLI, KATMAN); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval(); UYE_E.append(m)
mu_g = torch.stack([tahmin(m, gec)[0] for m in UYE_E]).mean(0); mu_a = torch.stack([tahmin(m, ara)[0] for m in UYE_E]).mean(0)
satir.append(dict(model='trained on 1902-1960 (5-member mean)', period=f'{AYAR["test_bas"]}-{YIL1}', **beceri(gec, mu_g)))
satir.append(dict(model='trained on 1902-1960 (5-member mean)', period=f'{AYAR["erken_son"]+1}-{AYAR["test_bas"]-1}', **beceri(ara, mu_a)))
pd.DataFrame(satir).to_csv(os.path.join(CIKTI, 'B1_erken_beceri.csv'), index=False)
b1.to_csv(os.path.join(CIKTI, 'B1_erken_uyeler.csv'), index=False)
del mu_g, mu_a; torch.cuda.empty_cache()
print(pd.DataFrame(satir).round(3).to_string(index=False))

# B2 skill against the main model, fold by fold (same folds, same seeds)
p3 = pd.read_csv(os.path.join(KAYNAK, 'P3_ana_kosu.csv')); p3 = p3[p3.model == AYAR['model']].set_index(['kat', 'tohum'])
k2 = b2.set_index(['kat', 'tohum']).join(p3[['r2_log', 'blok_ici_r_medyan', 'kapsama90']], rsuffix='_main', how='inner')
b2s = pd.DataFrame([dict(metric=m_, product_mean=float(k2[m_].mean()), product_min=float(k2[m_].min()), product_max=float(k2[m_].max()),
                         minimum_mean=float(k2[m_ + '_main'].mean()), minimum_min=float(k2[m_ + '_main'].min()), minimum_max=float(k2[m_ + '_main'].max()),
                         runs_product_higher=int((k2[m_] > k2[m_ + '_main']).sum()), n_runs=len(k2)) for m_ in ['r2_log', 'blok_ici_r_medyan', 'kapsama90']])
b2s.to_csv(os.path.join(CIKTI, 'B2_carpim_beceri.csv'), index=False); b2.to_csv(os.path.join(CIKTI, 'B2_carpim_uyeler.csv'), index=False)
print(b2s.round(3).to_string(index=False))

In [ ]:
# --- Attribution with the B1 and B2 ensembles (same method as Defter 21): every ring of 1902-2019 predicted with the
#     observed and the counterfactual climate; site-year means of the log difference ---
HALKA = ((R_yil >= YIL0 + 1) & (R_yil <= YIL_SON)).nonzero().squeeze(1)
NYa = YIL_SON - YIL0 + 1
ANAHTAR = R_site[HALKA] * NYa + (R_yil[HALKA] - YIL0)
SAYI = torch.bincount(ANAHTAR, minlength=S * NYa).float(); VAR = (SAYI.reshape(S, NYa) > 0).cpu().numpy()
def alan_yil(v): return (torch.bincount(ANAHTAR, weights=v.float(), minlength=S * NYa) / SAYI.clamp_min(1)).reshape(S, NYa).cpu().numpy().astype(np.float32)
def atif(modeller, etiket):
    kl = os.path.join(CIKTI, f'atif_{etiket}'); os.makedirs(kl, exist_ok=True); out = []
    for i, m in enumerate(modeller):
        f = os.path.join(kl, f'uye{i:02d}.npz')
        if not os.path.exists(f):
            global OZ
            OZ = OZ_F; muF = ongor(m, HALKA)[0]; OZ = OZ_C; muC = ongor(m, HALKA)[0]; OZ = OZ_F
            np.savez_compressed(f, fark_C=alan_yil(muF - muC)); del muF, muC; torch.cuda.empty_cache()
        out.append(np.load(f)['fark_C'])
    print(etiket, 'attribution members:', len(out))
    return np.stack(out)
UYE_C = []
for tohum in range(AYAR['n_tohum_carpim']):
    for kat in range(AYAR['n_kat']):
        kl = os.path.join(CIKTI, is_adi('yil', kat, AD_CARPIM, GIZLI, KATMAN, tohum))
        m = yap(AD_CARPIM, GIZLI, KATMAN); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval(); UYE_C.append(m)
FK_ana = {'x': np.stack([np.load(os.path.join(K21, 'atif', f'uye{i:02d}.npz'))['fark_C'] for i in range(len(MODELLER))
                                      if os.path.exists(os.path.join(K21, 'atif', f'uye{i:02d}.npz'))]),
      }
FK = {'main (minimum)': FK_ana['x'], 'trained 1902-1960': atif(UYE_E, 'erken'), 'product rule': atif(UYE_C, 'carpim')}
del FK_ana

def donem_ort(A, y0, y1, nmin=3):
    a, b = max(y0 - YIL0, 0), min(y1 - YIL0 + 1, NYa); v = VAR[:, a:b]; n = v.sum(1)
    return np.where(n >= nmin, (A[..., a:b] * v).sum(-1) / np.maximum(n, 1), np.nan)

In [ ]:
# --- B3. The three model families and the ring-based estimate on the same sites; 10-degree block intervals ---
LAT, LON = d['site_lat'].astype(float), d['site_lon'].astype(float)
BLOK = (np.floor((LAT + 90) / 10) * 100 + np.floor((LON + 180) / 10)).astype(int)
rngB = np.random.default_rng(2034)
def blok_boot(x, blok, fn, B=1000):
    ub = np.unique(blok); idx = {b: np.where(blok == b)[0] for b in ub}
    return np.nanpercentile([fn(x[np.concatenate([idx[b] for b in rngB.choice(ub, len(ub))])]) for _ in range(B)], [2.5, 97.5])
import glob as _glob
KAS = AYAR['analiz_son'] or next(iter(_glob.glob(os.path.join(ANA, '**', 'ANALIZ_SON'), recursive=True)), None)
Lr = np.full(S, np.nan)
if KAS and os.path.exists(os.path.join(KAS, 'N1a_uc_olay_kanali_alan.csv')):
    H = pd.read_csv(os.path.join(KAS, 'N1a_uc_olay_kanali_alan.csv'))
    kol_L = 'L_yuzde_yil' if 'L_yuzde_yil' in H.columns else next(c for c in H.columns if 'toplam' in c.lower())
    ix = H['studyCode'].astype(str).map({c: i for i, c in enumerate(soz['site_code'])}); ok_ = ix.notna().to_numpy()
    Lr[ix[ok_].astype(int).to_numpy()] = H.loc[ok_, kol_L].to_numpy(float)
E = {k_: donem_ort(A, 1990, YIL_SON, nmin=10) for k_, A in FK.items()}          # [members, S]
Em = {k_: yuzde(v.mean(0)) for k_, v in E.items()}
ortak = np.isfinite(Lr) & np.all([np.isfinite(v) for v in Em.values()], axis=0)
GR = {g: GRUP[g] for g in ['All sites', 'Coldest 25%', 'Driest 25%', 'Region: Europe', 'Region: North America', 'Region: Asia'] if g in GRUP}
GR.update({k_: v for k_, v in GRUP_CINS.items() if k_ in ('Genus: Fagus', 'Genus: Cedrus', 'Genus: Pseudotsuga', 'Genus: Larix')})
b3 = []
for g, msk in GR.items():
    m_ = msk & ortak
    if m_.sum() < 20: continue
    for kaynak, v in [*Em.items(), ('rings (no network)', Lr)]:
        x = v[m_]; lo_m, hi_m = blok_boot(x, BLOK[m_], np.mean); lo_d, hi_d = blok_boot(x, BLOK[m_], np.median)
        sat = dict(group=g, source=kaynak, n_sites=int(m_.sum()), mean_pct=float(x.mean()), mean_lo95=float(lo_m), mean_hi95=float(hi_m),
                   median_pct=float(np.median(x)), median_lo95=float(lo_d), median_hi95=float(hi_d))
        if kaynak in E:
            um = yuzde(np.nanmedian(E[kaynak][:, m_], axis=1)); sat.update(members_p05=float(np.quantile(um, 0.05)), members_p95=float(np.quantile(um, 0.95)))
        sat['site_r_with_main'] = float(np.corrcoef(x, Em['main (minimum)'][m_])[0, 1])
        sat['site_r_with_rings'] = float(np.corrcoef(x, Lr[m_])[0, 1])
        b3.append(sat)
b3 = pd.DataFrame(b3); b3.to_csv(os.path.join(CIKTI, 'B3_karsilastirma.csv'), index=False)
pd.set_option('display.width', 240)
print(b3[['group', 'source', 'n_sites', 'mean_pct', 'mean_lo95', 'mean_hi95', 'median_pct', 'site_r_with_main', 'site_r_with_rings']].round(2).to_string(index=False))

# chain-linked decadal index (sites present in consecutive decades) for each model family
DEC = list(range(1900, 2020, 10)); b3d = []
for k_, A in FK.items():
    ED = np.stack([donem_ort(A, max(dc, YIL0 + 1), dc + 9, nmin=3) for dc in DEC], -1); vd = np.isfinite(ED[0])
    for g in ['All sites', 'Coldest 25%', 'Driest 25%', 'Region: Europe']:
        msk = GRUP[g]; idx_u = None
        for jj, dc in enumerate(DEC):
            m_ = msk & vd[:, jj] & (vd[:, jj - 1] if jj else True)
            if m_.sum() < 15: break
            idx_u = np.median(ED[:, m_, 0], axis=1) if jj == 0 else idx_u + np.median(ED[:, m_, jj] - ED[:, m_, jj - 1], axis=1)
            b3d.append(dict(source=k_, group=g, decade=dc, n_sites=int(m_.sum()), median_pct=float(yuzde(np.median(idx_u))),
                            members_p05=float(yuzde(np.quantile(idx_u, 0.05))), members_p95=float(yuzde(np.quantile(idx_u, 0.95)))))
b3d = pd.DataFrame(b3d); b3d.to_csv(os.path.join(CIKTI, 'B3_onyil.csv'), index=False)
print(b3d[b3d.group == 'Driest 25%'].pivot_table(index='decade', columns='source', values='median_pct').round(2).to_string())
pd.DataFrame(dict(studyCode=soz['site_code'], rings_pct=Lr, **{k_: v for k_, v in Em.items()})).to_csv(os.path.join(CIKTI, 'B3_alan.csv'), index=False)

In [ ]:
print(sorted(f for f in os.listdir(CIKTI) if f.endswith('.csv')))
print('==== DEFTER 23B TAMAM ====')